In [1]:
%%capture
%pip install pymorphy3
%pip install wordcloud
%pip install catboost
%pip install google.colab
%pip install transformers datasets evaluate

In [3]:
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import nltk
import warnings
import joblib
import ast
import torch
import os
import evaluate
import gc

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

from catboost import CatBoostClassifier

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from pymorphy3 import MorphAnalyzer
from wordcloud import WordCloud

from collections import Counter

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from datasets import Dataset, load_from_disk

warnings.filterwarnings("ignore")

# Предобработка текстов

In [17]:
df = pd.read_csv('/content/drive/MyDrive/ ML HW3/data/dataset.csv')
df.head()

,id,url,title,subtitle,content,datetime,topic
0,2062742045,https://ria.ru/20251217/pdvsa-2062742045.html,PDVSA заявила о штатном режиме экспорта нефти,PDVSA заявила о штатном режиме экспорта нефти ...,Экспорт нефти и нефтепродуктов Венесуэлы осуще...,18:20 17.12.2025,1.0
1,2062740630,https://ria.ru/20251217/aziya-2062740630.html,Эксперт рассказал о новом этапе в сотрудничест...,Манчевич: сотрудничество России и Азии перешло...,Локализация высокотехнологичного производства ...,18:15 17.12.2025,1.0
2,2062738407,https://ria.ru/20251217/ssha-2062738407.html,США разрешили транспортировку нефти с проекта ...,"США разрешили финоперации, связанные с проекто...",Минфин США в среду разрешил финансовые операци...,18:09 17.12.2025,1.0
3,2062732693,https://ria.ru/20251217/roseksimbank-206273269...,"Кредитный продукт ""Деньги на экспорт"" от Росэк...","Кредитный продукт ""Деньги на экспорт"" от Росэк...","Кредитный продукт ""Деньги на экспорт"" от Росэк...",17:55 17.12.2025,1.0
4,2062732296,https://ria.ru/20251217/eksar-2062732296.html,"Агентство ""Эксар"" подвело итоги работы с банка...","Агентство ""Эксар"" провело итоговое в 2025 г за...",Российское агентство по страхованию экспортных...,17:54 17.12.2025,1.0


In [22]:
building_df = df[df['topic'] == 6]['content']
building_df.sample(5).tolist()

['Фото: AP Александра Лисица Президент Ботсваны Мокветси Масиси поразился найденному в его стране огромному алмазу размером в 2492 карата и попал на видео. Об этом пишет Sky News. Кадры, на которых Масиси бурно реагирует на находку, разлетелись по мировым СМИ и соцсетям — глава государства очень удивился весу и размерам камня и долго осматривал его. «Это ошеломляет. Мне повезло увидеть это», — сказал Масиси. Почти полукилограммовый алмаз — второй по размерам в истории человечества. Самый большой камень — весом 3106 карат (621 граммов) — был найден в Южной Африке в 1905 году. Ботсвана — второй по величине производитель природных алмазов в мире после России. Ранее в Индии рабочий нашел в шахте алмаз, стоимость которого оценивают в 74 тысячи фунтов стерлингов (восемь миллионов рублей). Он планирует потратить компенсацию от государства на выплату долга, строительство нового дома и помощь семье.',
 'Фото: Кирилл Каллиников / РИА Новости Нина Ташевская Председатель комитета Госдумы по строит

# Векторизация + обучение моделей

## Fine-tuned BERT

In [5]:
df = pd.read_csv('/content/drive/MyDrive/ ML HW3/data/dataset.csv')
df['text'] = df['content'].fillna('')
df.head()

,id,url,title,subtitle,content,datetime,topic,text
0,2062742045,https://ria.ru/20251217/pdvsa-2062742045.html,PDVSA заявила о штатном режиме экспорта нефти,PDVSA заявила о штатном режиме экспорта нефти ...,Экспорт нефти и нефтепродуктов Венесуэлы осуще...,18:20 17.12.2025,1.0,Экспорт нефти и нефтепродуктов Венесуэлы осуще...
1,2062740630,https://ria.ru/20251217/aziya-2062740630.html,Эксперт рассказал о новом этапе в сотрудничест...,Манчевич: сотрудничество России и Азии перешло...,Локализация высокотехнологичного производства ...,18:15 17.12.2025,1.0,Локализация высокотехнологичного производства ...
2,2062738407,https://ria.ru/20251217/ssha-2062738407.html,США разрешили транспортировку нефти с проекта ...,"США разрешили финоперации, связанные с проекто...",Минфин США в среду разрешил финансовые операци...,18:09 17.12.2025,1.0,Минфин США в среду разрешил финансовые операци...
3,2062732693,https://ria.ru/20251217/roseksimbank-206273269...,"Кредитный продукт ""Деньги на экспорт"" от Росэк...","Кредитный продукт ""Деньги на экспорт"" от Росэк...","Кредитный продукт ""Деньги на экспорт"" от Росэк...",17:55 17.12.2025,1.0,"Кредитный продукт ""Деньги на экспорт"" от Росэк..."
4,2062732296,https://ria.ru/20251217/eksar-2062732296.html,"Агентство ""Эксар"" подвело итоги работы с банка...","Агентство ""Эксар"" провело итоговое в 2025 г за...",Российское агентство по страхованию экспортных...,17:54 17.12.2025,1.0,Российское агентство по страхованию экспортных...


In [6]:
X = df['text']
y = df['topic'].apply(lambda x: int(x))

In [7]:
seq_len = [len(str(i).split()) for i in df['text']]
max_seq_len = max(seq_len)
max_seq_len

7850

In [9]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

Train: 30870, Val: 5145, Test: 5146


In [10]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

model_name = "DeepPavlov/rubert-base-cased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=9,
    problem_type="single_label_classification"
)

model.to(device)

GPU: Tesla T4
Memory: 15.8 GB


tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/714M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at DeepPavlov/rubert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1

In [11]:
train_dataset = Dataset.from_dict({
    'text': X_train.tolist(),
    'label': [int(x) for x in y_train.tolist()]
})

val_dataset = Dataset.from_dict({
    'text': X_val.tolist(),
    'label': [int(x) for x in y_val.tolist()]
})

test_dataset = Dataset.from_dict({
    'text': X_test.tolist(),
    'label': [int(x) for x in y_test.tolist()]
})

print(f"Train dataset: {train_dataset}")
print(f"Val dataset: {val_dataset}")

Train dataset: Dataset({
    features: ['text', 'label'],
    num_rows: 30870
})
Val dataset: Dataset({
    features: ['text', 'label'],
    num_rows: 5145
})


In [12]:
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=512
    )

In [13]:
train_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=['text'],
    desc="Tokenizing train"
)
train_dataset.save_to_disk('data/tokenized/train')

val_dataset = val_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=['text'],
    desc="Tokenizing val"
)
val_dataset.save_to_disk('data/tokenized/val')

test_dataset = test_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=['text'],
    desc="Tokenizing test"
)
test_dataset.save_to_disk('data/tokenized/test')

Tokenizing train:   0%|          | 0/30870 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/30870 [00:00<?, ? examples/s]

Tokenizing val:   0%|          | 0/5145 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5145 [00:00<?, ? examples/s]

Tokenizing test:   0%|          | 0/5146 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5146 [00:00<?, ? examples/s]

In [14]:
del train_dataset
del val_dataset
del test_dataset

gc.collect()

84

In [15]:
train_dataset = load_from_disk('data/tokenized/train')
val_dataset = load_from_disk('data/tokenized/val')
test_dataset = load_from_disk('data/tokenized/test')

train_dataset.set_format('torch')
val_dataset.set_format('torch')
test_dataset.set_format('torch')

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")




Train: 30870, Val: 5145, Test: 5146


In [16]:
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

In [ ]:
training_args = TrainingArguments(
    output_dir='./rubert_finetuned',

    # Эпохи и батчи
    num_train_epochs=6,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    # Learning rate и scheduler
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',

    # Оценка и сохранение
    eval_strategy='steps',
    eval_steps=200,
    save_strategy='steps',
    save_steps=200,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    greater_is_better=True,

    # Оптимизации
    fp16=torch.cuda.is_available(),
    gradient_accumulation_steps=2,
    dataloader_num_workers=2,

    # Логирование
    logging_dir='./logs',
    logging_steps=100,
    report_to='none',  # отключаем wandb и т.д.

    # Seed
    seed=42,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

In [ ]:
print("Начинаем обучение...")
train_result = trainer.train()

# %%
# Выводим результаты обучения
print(f"\nРезультаты обучения:")
print(f"Training loss: {train_result.training_loss:.4f}")
print(f"Training time: {train_result.metrics['train_runtime']:.1f} sec")

# %%
# Оцениваем на валидации
val_results = trainer.evaluate()
print(f"\nValidation Accuracy: {val_results['eval_accuracy']:.4f}")

# %%
# Оцениваем на тесте
test_results = trainer.evaluate(test_dataset)
print(f"\nTest Accuracy: {test_results['eval_accuracy']:.4f}")

Начинаем обучение...


Step,Training Loss,Validation Loss,Accuracy
200,1.566800,1.247127,0.632329
400,0.834100,0.666714,0.806122
600,0.615200,0.544744,0.817784
800,0.482400,0.477027,0.850988
1000,0.437000,0.381726,0.877551
1200,0.370600,0.344143,0.879495
1400,0.372600,0.331189,0.893748
1600,0.347200,0.324872,0.892128
1800,0.329000,0.312960,0.897311
2000,0.245000,0.415578,0.872044



Результаты обучения:
Training loss: 0.3044
Training time: 3815.9 sec



Validation Accuracy: 0.9316

Test Accuracy: 0.9257


In [ ]:
predictions = trainer.predict(test_dataset)
y_pred = np.argmax(predictions.predictions, axis=-1)

topic_names = {
    0: 'Общество/Россия',
    1: 'Экономика',
    2: 'Силовые структуры',
    3: 'Бывший СССР',
    4: 'Спорт',
    5: 'Забота о себе',
    6: 'Строительство',
    7: 'Туризм/Путешествия',
    8: 'Наука и техника'
}

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=[topic_names[i] for i in range(9)]
))


Classification Report:
                    precision    recall  f1-score   support

   Общество/Россия       0.91      0.91      0.91      1355
         Экономика       0.92      0.94      0.93      1975
 Силовые структуры       0.92      0.94      0.93       385
       Бывший СССР       0.93      0.95      0.94       678
             Спорт       0.99      0.99      0.99       730
     Забота о себе       0.97      0.98      0.97       174
     Строительство       0.82      0.77      0.79       511
Туризм/Путешествия       0.93      0.92      0.92        95
   Наука и техника       0.96      0.92      0.94       272

          accuracy                           0.93      6175
         macro avg       0.93      0.92      0.93      6175
      weighted avg       0.93      0.93      0.93      6175



In [ ]:
trainer.save_model('./rubert_finetuned_best')
tokenizer.save_pretrained('./rubert_finetuned_best')

('./rubert_finetuned_best/tokenizer_config.json',
 './rubert_finetuned_best/special_tokens_map.json',
 './rubert_finetuned_best/vocab.txt',
 './rubert_finetuned_best/added_tokens.json',
 './rubert_finetuned_best/tokenizer.json')

In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
model_path = './rubert_finetuned_best'

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.to(device)
model.eval()

X_kaggle = pd.read_csv('test_news.csv')
X_kaggle['text'] = X_kaggle['content'].fillna('')

The tokenizer you are loading from './rubert_finetuned_best' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


In [ ]:
def predict_batch(texts, model, tokenizer, device, batch_size=32):
    """
    Предсказания батчами для экономии памяти
    """
    model.eval()
    all_predictions = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]

        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors='pt'
        )

        input_ids = encoded['input_ids'].to(device)
        attention_mask = encoded['attention_mask'].to(device)

        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            predictions = torch.argmax(outputs.logits, dim=-1)
            all_predictions.extend(predictions.cpu().numpy())

        if (i // batch_size) % 20 == 0:
            print(f"Processed {min(i + batch_size, len(texts))}/{len(texts)}")

    return np.array(all_predictions)

In [ ]:
kaggle_texts = X_kaggle['text'].tolist()
kaggle_predictions = predict_batch(kaggle_texts, model, tokenizer, device, batch_size=32)

Processed 32/26275
Processed 672/26275
Processed 1312/26275
Processed 1952/26275
Processed 2592/26275
Processed 3232/26275
Processed 3872/26275
Processed 4512/26275
Processed 5152/26275
Processed 5792/26275
Processed 6432/26275
Processed 7072/26275
Processed 7712/26275
Processed 8352/26275
Processed 8992/26275
Processed 9632/26275
Processed 10272/26275
Processed 10912/26275
Processed 11552/26275
Processed 12192/26275
Processed 12832/26275
Processed 13472/26275
Processed 14112/26275
Processed 14752/26275
Processed 15392/26275
Processed 16032/26275
Processed 16672/26275
Processed 17312/26275
Processed 17952/26275
Processed 18592/26275
Processed 19232/26275
Processed 19872/26275
Processed 20512/26275
Processed 21152/26275
Processed 21792/26275
Processed 22432/26275
Processed 23072/26275
Processed 23712/26275
Processed 24352/26275
Processed 24992/26275
Processed 25632/26275
Processed 26272/26275


In [ ]:
submission = pd.DataFrame({
    'index': range(len(kaggle_predictions)),
    'topic': kaggle_predictions
})

submission.to_csv('rubert_finetuned.csv', index=False)

In [ ]:
submission.shape

(26275, 2)

# Предсказание для Kaggle

##### Предобработка инпута для Kaggle

##### Получение предсказаний